In [30]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture as GMM
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import homogeneity_completeness_v_measure
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import sklearn.datasets as datasets
import ydata_profiling as yp
from sklearn.preprocessing import StandardScaler

## Carga y exploración inicial

In [35]:
df=pd.DataFrame(datasets.load_wine().data,columns=datasets.load_wine().feature_names)

In [36]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


## Preprocesamiento

In [37]:
scaler=StandardScaler()
df_scaled=scaler.fit_transform(df)

In [43]:
df_scaled[0:5,:]

array([[ 1.51861254, -0.5622498 ,  0.23205254, -1.16959318,  1.91390522,
         0.80899739,  1.03481896, -0.65956311,  1.22488398,  0.25171685,
         0.36217728,  1.84791957,  1.01300893],
       [ 0.24628963, -0.49941338, -0.82799632, -2.49084714,  0.01814502,
         0.56864766,  0.73362894, -0.82071924, -0.54472099, -0.29332133,
         0.40605066,  1.1134493 ,  0.96524152],
       [ 0.19687903,  0.02123125,  1.10933436, -0.2687382 ,  0.08835836,
         0.80899739,  1.21553297, -0.49840699,  2.13596773,  0.26901965,
         0.31830389,  0.78858745,  1.39514818],
       [ 1.69154964, -0.34681064,  0.4879264 , -0.80925118,  0.93091845,
         2.49144552,  1.46652465, -0.98187536,  1.03215473,  1.18606801,
        -0.42754369,  1.18407144,  2.33457383],
       [ 0.29570023,  0.22769377,  1.84040254,  0.45194578,  1.28198515,
         0.80899739,  0.66335127,  0.22679555,  0.40140444, -0.31927553,
         0.36217728,  0.44960118, -0.03787401]])

In [42]:
pca=PCA(n_components=5)
df_pca=pca.fit_transform(df_scaled)

In [44]:
df_pca[0:5,:]

array([[ 3.31675081,  1.44346263, -0.16573904, -0.21563119,  0.69304284],
       [ 2.20946492, -0.33339289, -2.02645737, -0.29135832, -0.25765463],
       [ 2.51674015,  1.0311513 ,  0.98281867,  0.72490231, -0.25103312],
       [ 3.75706561,  2.75637191, -0.17619184,  0.56798331, -0.31184159],
       [ 1.00890849,  0.86983082,  2.02668822, -0.40976579,  0.2984575 ]])

## Modelos

In [ ]:
# Modelos de clustering rigidos (KMeans clasico, KMeans++ y MiniBatchKMeans)
def fit_all(X, k):
    models = {
        "KMeans": KMeans(n_clusters=k, n_init=10, random_state=42),
        "KMeans++": KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42),
        "MiniBatchKMeans": MiniBatchKMeans(n_clusters=k, n_init=10, random_state=42),
    }
    out = {}
    for name, m in models.items():
        m.fit(X)
        labels = m.labels_ if hasattr(m, "labels_") else m.predict(X)
        sil = silhouette_score(X, labels)
        extra = {}
        if isinstance(m, GMM):
            extra["BIC"] = m.bic(X)
            extra["AIC"] = m.aic(X)
            extra["loglike"] = m.score(X) * len(X)
        else:
            extra["inertia"] = m.inertia_
        out[name] = {"model": m, "labels": labels, "silhouette": sil, **extra}
    return out